# Supervised Fine-Tuning

Example: to train a foundation `Qwen2.5-0.5B` for solving maths problems

## Setup

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth" "trl>=0.14" "peft>=0.13" "transformers>=4.45" accelerate datasets bitsandbytes unsloth_zoo

  Cloning https://github.com/unslothai/unsloth to /tmp/pip-install-_7e6l2_9/unsloth_9aff0a5f4e1445cca405ff40a00c2e54
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth /tmp/pip-install-_7e6l2_9/unsloth_9aff0a5f4e1445cca405ff40a00c2e54
  Resolved https://github.com/unslothai/unsloth to commit 2a95eb78bcd4207e5a79cd48a8e0ab81f8272214
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.9.2-py3-none-any.whl size=43772343 sha256=c7b6884ecd45a341c712a19f02d5c75a9215b9ab776c746d8b4d4887a33c2d9b
  Stored in directory: /tmp/pip-ephem-wheel-cache-7_3bnntq/wheels/94/bc/64/a1c9ffd137a50b590a95fb24d69298c91b05afbf40b5cfe283
Successfully built unsloth


This block sets up the core components for model loading and fine-tuning:
*   **Imports**: Essential libraries like `gc` (garbage collector), `torch` (PyTorch), `datasets`, and `unsloth` are imported.
*   **Global Variables**: `MODEL_ID` specifies the base model to be used (Qwen/Qwen2.5-0.5B-Instruct), `MAX_SEQ_LEN` defines the maximum sequence length for the model, and `USE_4BIT` enables 4-bit quantization for memory efficiency.
*   **`load_base_model()`**: This function loads the pre-trained language model and its tokenizer using `FastLanguageModel.from_pretrained`. It supports 4-bit quantization via `load_in_4bit=True` to reduce memory footprint.
*   **`attach_lora()`**: This function applies LoRA (Low-Rank Adaptation) adapters to the model. LoRA is a parameter-efficient fine-tuning technique that reduces the number of trainable parameters, making fine-tuning faster and less memory-intensive. It configures the LoRA rank, target modules, alpha, dropout, and uses Unsloth's gradient checkpointing for further VRAM savings.
*   **`cleanup()`**: A utility function to free up GPU memory by calling `gc.collect()` and `torch.cuda.empty_cache()`.

In [ ]:
import gc
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-0.5B"   # swap for any Unsloth-supported model
MAX_SEQ_LEN = 2048                         # Unsloth supports up to ~32k context

# Unsloth turns a lossy 4-bit model back into a usable model via:
#   nf4 quantized weights + bfloat16 compute + double-quant + Flash attention.
# You don't configure bitsandbytes yourself — just flip load_in_4bit=True.
USE_4BIT = True

def load_base_model():
    """Load the base instruct model + tokenizer, with QLoRA 4-bit if enabled."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,             # None => auto-pick (bf16 if supported, else fp16)
        load_in_4bit=USE_4BIT,
    )
    return model, tokenizer

def attach_lora(model):
    """Attach LoRA adapters. Unsloth fills in arch-specific target modules
    for you, but for full coverage (attention + MLP) we list them explicitly.
    Returns the LoRA-fitted model (in-place via PEFT)."""
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,                  # LoRA rank. 8/16/32/64; bigger = more capacity + VRAM.
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=32,          # rule of thumb: alpha = 2*r.
        lora_dropout=0.05,
        bias="none",            # train no bias terms.
        use_gradient_checkpointing="unsloth",  # Unsloth's trick: ~30% less VRAM, no slowdown.
        random_state=3407,
        use_rslora=False,      # rsLoRA helps for high-rank; leave off for small r.
        loftq_config=None,
    )
    return model

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
model, tokenizer = load_base_model()
model = attach_lora(model)

# Unsloth's get_chat_template wraps apply_chat_template and maps your
# dataset's keys to the model's template. "chatml" matches Qwen/Llama3.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",   # also: llama-3, zephyr, mistral, vicuna, alpaca, ...
    map_eos_token=True,
)

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_r6epwbyc/tokenizer_config.json.


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

## Import Dataset

Dataset: [`HuggingFaceH4/MATH-500`](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)

Schema:
```
  problem  : the question text (LaTeX)
  solution : the worked solution (LaTeX), ending in \boxed{<answer>}
  answer   : the ground-truth final answer string (e.g. "p - q", "3")
  subject  : e.g. "Precalculus", "Intermediate Algebra"
  level    : 1..5 difficulty
  unique_id: e.g. "test/precalculus/807.json"
```

This section prepares the model and data for Supervised Fine-Tuning (SFT).
*   **Dataset Loading**: The `HuggingFaceH4/MATH-500` dataset is loaded, which consists of 500 competition math problems. The `split="test"` argument is used to load the specific test split.
*   **System Prompt**: `SYSTEM` defines the role of the AI as a "careful math problem solver" and specifies the desired output format (final answer inside `\boxed{...}`).
*   **`to_sft_rows()` Function**: This function transforms the raw dataset into a format suitable for Supervised Fine-Tuning (SFT). It structures each example as a list of messages (system, user problem, assistant solution) in a chat-like format. The assistant's content includes the full worked solution.
*   **Dataset Creation**: The processed `sft_rows` are converted into a `datasets.Dataset` object and then split into training and testing sets with a `test_size` of 10% and a fixed `seed` for reproducibility.

In [ ]:
from datasets import load_dataset

math500 = load_dataset("HuggingFaceH4/MATH-500", split="test")

SYSTEM = (
    "You are a careful math problem solver. Read the problem, reason step "
    "by step, then give the final answer inside \\boxed{...}."
)

def to_sft_rows(ds):
    """Map MATH-500 -> {"messages": [user(problem), assistant(solution)]}.
    Each assistant target is the full worked solution ending in \\boxed{answer}."""
    out = []
    for ex in ds:
        out.append({
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": ex["problem"]},
                {"role": "assistant", "content": ex["solution"]},
            ],
        })
    return out

def fmt_sft(examples):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples["messages"]
    ]
    return {"text": texts}

sft_rows = to_sft_rows(math500)
sft_ds = Dataset.from_list(sft_rows).train_test_split(test_size=0.1, seed=42)
sft_ds = sft_ds.map(fmt_sft, batched=True)

In [ ]:
# sample
sft_rows[0]

{'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n\n[asy]\nunitsize(0.8 cm);\n\ndraw((-0.5,0)--(3.5,0));\ndraw((0,-0.5)--(0,3.5));\ndraw(arc((0,0),3,0,90),red,Arrow(6));\n\ndot((0,3), red);\nlabel("$(0,3)$", (0,3), W);\ndot((3,0), red);\n[/asy]\n\nTherefore, the polar coordinates are $\\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}.$',
 'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'subject': 'Precalculus',
 'level': 2,
 'unique_id': 'test/precalculus/807.json'}

## SFT (Supervised Fine-Tuning)

This cell configures and runs the Supervised Fine-Tuning (SFT) process:
*   **`SFTConfig`**: This object defines the training parameters, such as:
    *   `output_dir`: Where to save the fine-tuned model and logs.
    *   `num_train_epochs`: Number of passes over the training data.
    *   `per_device_train_batch_size`, `gradient_accumulation_steps`: Control batch size and memory usage.
    *   `learning_rate`, `lr_scheduler_type`, `warmup_ratio`: Learning rate and scheduling strategy.
    *   `logging_steps`, `save_strategy`: How often to log and save the model.
    *   `bf16`/`fp16`: Enables BFloat16 or Float16 for faster training on compatible GPUs.
    *   `optim`: Optimizer to use (`adamw_8bit` for memory efficiency).
    *   `weight_decay`, `seed`: Regularization and reproducibility settings.
    *   `max_seq_length`: Maximum sequence length for training examples.
    *   `packing`: When `True`, packs multiple short sequences into a single input to optimize GPU utilization (set to `False` here).
*   **`SFTTrainer`**: Initializes the trainer with the model, tokenizer, training/evaluation datasets, and the configured training arguments.
*   **`sft_trainer.train()`**: Starts the fine-tuning process. Unsloth patches TRL's `SFTTrainer` to use its optimized kernels for faster training.

In [ ]:
# Unsloth patches TRL's SFTTrainer so it uses its fast kernels. Pass
# dataset_text_field="text" (we created it above) OR pass the raw
# "messages" field and let the trainer apply the template — both work.
sft_cfg = SFTConfig(
    output_dir="out/sft",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,             # Unsloth/LoRA rule: ~10x the full-FT rate.
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=2,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",             # paged 8-bit AdamW -> more VRAM headroom.
    weight_decay=0.01,
    seed=3407,
    # MATH-500 solutions (LaTeX) run longer than the toy QA rows; keep the
    # full context. If you OOM, lower MAX_SEQ_LEN (2048->1024) or the batch.
    max_seq_length=MAX_SEQ_LEN,
    packing=False,                 # True packs short seqs -> big speedup on long data.
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_ds["train"],
    eval_dataset=sft_ds["test"],
    args=sft_cfg,
)
sft_trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 339
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
2,1.572487
4,1.536218
6,1.425977
8,1.461913
10,1.102676
12,1.038276
14,0.783585
16,0.804578
18,0.712590
20,0.891220


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=339, training_loss=0.6712910211191768, metrics={'train_runtime': 363.6776, 'train_samples_per_second': 3.712, 'train_steps_per_second': 0.932, 'total_flos': 1326038431672320.0, 'train_loss': 0.6712910211191768, 'epoch': 3.0})

In [ ]:

model.save_pretrained("out/sft_adapter")          # adapter only
tokenizer.save_pretrained("out/sft_adapter")
# model.save_pretrained_merged("out/sft_merged", tokenizer, save_method="merged_16bit")
print("SFT done -> adapter saved to out/sft_adapter")

SFT done -> adapter saved to out/sft_adapter


In [ ]:
# Flush memory
del sft_trainer, model, tokenizer
cleanup()

## Inference

This section demonstrates how to use the fine-tuned model for inference. The goal is to observe if the model, after SFT, can better follow instructions and perform the tasks it was trained on.

In [ ]:
# Load model with trained adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="out/sft_adapter", # YOUR MODEL YOU TRAINED
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=USE_4BIT,
)

FastLanguageModel.for_inference(model)
model.eval()

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [ ]:
# Prepare a new problem for inference
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "What is the value of $x-2$ if $2x+5=11$?"},
]

# Apply the chat template and tokenize
chat_template_messages = [messages]

inputs = tokenizer(
    [tokenizer.apply_chat_template(x, tokenize=False, add_generation_prompt=True) for x in chat_template_messages],
    return_tensors="pt"
).to("cuda")

# Generate a response
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

# Decode and print the output
response = tokenizer.batch_decode(outputs)
print(response[0])

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are a careful math problem solver. Read the problem, reason step by step, then give the final answer inside \boxed{...}.<|im_end|>
<|im_start|>user
What is the value of $x-2$ if $2x+5=11$?<|im_end|>
<|im_start|>assistant
We can solve for $x$ by subtracting 5 from both sides of the equation $2x+5=11$ to get $2x=6$.  Then we divide both sides by 2 to get $x=3$.  Now we can find $x-2$ by substituting 3 for $x$ to get $3-2=\boxed{1}$.<|im_end|>
